Project performaces from Devman

This notebook analyzes project performances from Devamn

In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

loading and preparing data

In [52]:


df = pd.read_excel(
    "EIR Donations 2 (2).xlsx",
    sheet_name="development office"
)

# # Clean donor type
# df["Indivisual/Organisation"] = (
#     df["Indivisual/Organisation"]
#     .astype(str)
#     .str.strip()
#     .str.lower()
# )

# Keep individual donors only
df = df[df["Indivisual/Organisation"] == "organisation"]

# Clean donor names
df["Paid from/Full name"] = (
    df["Paid from/Full name"]
    .astype(str)
    .str.strip()
)

# Ensure year is numeric
df["years"] = pd.to_numeric(df["years"], errors="coerce")

# Keep only 2024 and 2025
df = df[df["years"].isin([2024, 2025])]


In [53]:
df.head(2)

,Indivisual/Organisation,Paid from/Full name,Date,years,months,cleaned projects,include,Project/Project name,Amount,Project/Parent name/Full name
355,organisation,Transport Education Training Authority SETA,2024-03-18,2024,March,Bursaries - FEBE,Yes,Bursaries - FEBE,7800000.0,Faculty of Engineering & the Built Environment
356,organisation,"Media, Information and Communication Technolog...",2024-01-20,2024,January,Bursaries - Non Specific,Yes,Bursaries - Non Specific,3414990.4,Development Office


Define Donor

In [54]:
donors_2024 = set(
    df[df["years"] == 2024]["Paid from/Full name"]
)

donors_2025 = set(
    df[df["years"] == 2025]["Paid from/Full name"]
)

Donor Counts

In [55]:
unique_donors_2024 = len(donors_2024)
unique_donors_2025 = len(donors_2025)

# Recurring donors (appear in both years)
recurring_donors = donors_2024.intersection(donors_2025)

# Gained donors (new in 2025, not in 2024)
donors_gained = donors_2025.difference(donors_2024)

# Lost donors (were in 2024, missing in 2025)
donors_lost = donors_2024.difference(donors_2025)


retention,churn, aquisition

In [56]:
retention_rate = len(recurring_donors) / unique_donors_2024 if unique_donors_2024 > 0 else 0
churn_rate = len(donors_lost) / unique_donors_2024 if unique_donors_2024 > 0 else 0
acquisition_rate = len(donors_gained) / unique_donors_2025 if unique_donors_2025 > 0 else 0





In [57]:
project_matrix = (
    df.groupby(["cleaned projects", "years"])
      .agg(
          total_amount=("Amount", "sum"),
          donor_count=("Paid from/Full name", "nunique"),
          avg_gift=("Amount", "mean"),
          donation_count=("Amount", "count")
      )
      .reset_index()
)

project_matrix


,cleaned projects,years,total_amount,donor_count,avg_gift,donation_count
0,APK Teaching Excellence,2025,3.045500e+07,1,3.045500e+07,1
1,Bursaries,2025,3.000000e+06,1,3.000000e+06,1
2,Bursaries - CBE,2024,2.018184e+07,15,1.062202e+06,19
3,Bursaries - CBE,2025,3.291895e+07,12,1.431259e+06,23
4,Bursaries - Education,2024,3.473558e+06,2,1.736779e+06,2
5,Bursaries - Education,2025,1.800347e+07,10,1.500289e+06,12
6,Bursaries - FADA,2024,5.369600e+06,1,1.789867e+06,3
7,Bursaries - FADA,2025,2.460000e+06,2,1.230000e+06,2
8,Bursaries - FEBE,2024,1.306598e+08,18,6.876830e+06,19
9,Bursaries - FEBE,2025,1.841688e+07,10,1.841688e+06,10


Average Size

In [58]:
avg_gift_2024 = df[df["years"] == 2024]["Amount"].mean()
avg_gift_2025 = df[df["years"] == 2025]["Amount"].mean()


In [59]:
donor_summary = pd.DataFrame({
    "Metric": [
        "Unique Donors (2024)",
        "Unique Donors (2025)",
        "Recurring Donors",
        "Donors Gained (2025)",
        "Donors Lost (2025)",
        "Retention Rate",
        "Churn Rate",
        "Acquisition Rate",
        "Average Gift Size (2024)",
        "Average Gift Size (2025)"
    ],
    "Value": [
        unique_donors_2024,
        unique_donors_2025,
        len(recurring_donors),
        len(donors_gained),
        len(donors_lost),
        round(retention_rate * 100, 2),
        round(churn_rate * 100, 2),
        round(acquisition_rate * 100, 2),
        round(avg_gift_2024, 2),
        round(avg_gift_2025, 2)
    ]
})

donor_summary


,Metric,Value
0,Unique Donors (2024),92.00
1,Unique Donors (2025),56.00
2,Recurring Donors,22.00
3,Donors Gained (2025),34.00
4,Donors Lost (2025),70.00
5,Retention Rate,23.91
6,Churn Rate,76.09
7,Acquisition Rate,60.71
8,Average Gift Size (2024),2398863.56
9,Average Gift Size (2025),3577564.72


In [60]:
donor_summary.to_excel(
    "indivisual_Donor_Metrics_2024_vs_2025.xlsx",
    index=False
)
